In [1]:
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.transforms import Compose, ToTensor, Normalize, RandomHorizontalFlip
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# 设置matplotlib为黑色主题
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': 'black',
    'axes.facecolor': 'black',
    'axes.edgecolor': 'white',
    'axes.labelcolor': 'white',
    'text.color': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'grid.color': '#444444',
    'grid.alpha': 0.3,
    'legend.facecolor': 'black',
    'legend.edgecolor': 'white',
})

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# <font color="#00FFFF" >**第六章 CNN高级主题与应用**</font>

## **本章内容**

> 1. CNN神经网络训练完整实践（RPS图像分类）
> 2. 现代卷积神经网络（VGG / GoogLeNet / ResNet）
> 3. 迁移学习（Transfer Learning）


##  <font color="#FFEA00" >**第1节 CNN神经网络训练完整实践**</font>

### **<font color="#39FF14" size=6 >一、使用ImageFolder加载数据集</font>**

#### <font color="#CCFF00">**ImageFolder简介**</font>

`ImageFolder`是PyTorch提供的通用数据集类，用于加载<font color="#f6ff00">**按文件夹组织**</font>的自定义图像数据集。

**文件夹结构要求**：
```
dataset_root/  ← 根目录
├── class_a/   ← 类别A的文件夹
│   ├── img1.png
│   ├── img2.png
│   └── ...
├── class_b/   ← 类别B的文件夹
│   ├── img1.png
│   └── ...
└── class_c/   ← 类别C的文件夹
    └── ...
```

<font color="#FF00A0" >**关键特点**</font>：
- 每个子文件夹的名称即为类别标签
- 自动将文件夹名映射为类别索引（0, 1, 2, ...）
- 支持常见的图像格式（PNG, JPG, JPEG等）
- 与`transform`结合使用，方便数据预处理

In [2]:
# 下载Rock Paper Scissors数据集
import requests
import zipfile
import os
import errno

def download_rps(localfolder=''):
    '''
    下载Rock Paper Scissors数据集
    包含2520张训练图像和372张测试图像
    每张图像为300x300像素的RGBA格式
    '''
    filenames = ['rps.zip', 'rps-test-set.zip']
    for filename in filenames:
        try:
            os.mkdir(f'{localfolder}{filename[:-4]}')
            localfile = f'{localfolder}{filename}'
            url = 'https://storage.googleapis.com/download.tensorflow.org/data/{}'
            print(f'正在下载 {filename}...')
            r = requests.get(url.format(filename), allow_redirects=True)
            open(localfile, 'wb').write(r.content)
            print(f'解压 {filename}...')
            with zipfile.ZipFile(localfile, 'r') as zip_ref:
                zip_ref.extractall(localfolder)
            print(f'{filename} 下载完成！')
        except OSError as e:
            if e.errno != errno.EEXIST:
                raise
            else:
                print(f'{filename[:-4]} 文件夹已存在，跳过下载')

# 下载数据集（在当前目录创建rps和rps-test-set文件夹）
download_rps()

rps 文件夹已存在，跳过下载
rps-test-set 文件夹已存在，跳过下载


#### <font color="#CCFF00">**Rock Paper Scissors数据集</font>**

**数据集来源**：Laurence Moroney (lmoroney@gmail.com)

**许可证**：Creative Commons (CC BY 2.0)

**数据集特点**：
- 包含2,892张手势图像（石头、剪刀、布）
- 使用CGI技术生成的合成数据集
- 每张图像300×300像素，RGBA四通道格式
- 训练集：2,520张，测试集：372张
- 三个类别完全平衡，每类840张训练图像


In [3]:
# ImageFolder使用示例
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor

# 第一步：创建临时数据集（仅用于计算统计量）
temp_transform = Compose([
    Resize(28),      # 调整大小为28x28
    ToTensor()       # 转为Tensor，自动归一化到[0,1]
])

# 假设数据集已下载到rps文件夹
temp_dataset = ImageFolder(root='rps', transform=temp_transform)

# 查看第一个样本
print(f'图像形状: {temp_dataset[0][0].shape}')  # torch.Size([3, 28, 28])
print(f'类别标签: {temp_dataset[0][1]}')       # 0, 1, 或 2
print(f'类别名称: {temp_dataset.classes}')     # ['paper', 'rock', 'scissors']

print('ImageFolder会自动将子文件夹名映射为类别标签')
print('类别索引: 0=paper, 1=rock, 2=scissors')

图像形状: torch.Size([3, 28, 28])
类别标签: 0
类别名称: ['paper', 'rock', 'scissors']
ImageFolder会自动将子文件夹名映射为类别标签
类别索引: 0=paper, 1=rock, 2=scissors


### **<font color="#39FF14" size=6 >二、CNN神经网络训练完整流程</font>**

#### <font color="#CCFF00">**完整实战：RPS（石头剪刀布）图像分类**</font>

本节将以**RPS（Rock-Paper-Scissors）数据集**为例，展示一个完整的CNN训练流程，涵盖数据下载、预处理、模型设计、训练技巧等所有关键环节。

<font color="#FF00A0" >**RPS数据集简介**</font>：
- 类别数：3类（石头、剪刀、布）
- 图像尺寸：300×300像素
- 训练集：约2520张图像
- 测试集：约372张图像

#### <font color="#CCFF00">**Step 1：数据下载与ImageFolder加载**</font>

<font color="#FF00A0" >**数据下载**</font>：使用自定义函数从TensorFlow官方仓库下载RPS数据集

<font color="#FF00A0" >**ImageFolder加载**</font>：PyTorch的`ImageFolder`自动识别文件夹结构，将子文件夹名作为类别标签

In [ ]:
# ============================================================================
# Step 1: 数据下载与ImageFolder加载
# ============================================================================

import requests
import zipfile
import os
import errno

# 避免重复下载：如果文件夹已存在，则跳过下载

def download_rps(localfolder=''):
    """
    下载RPS数据集（石头剪刀布）
    
    Args:
        localfolder: 下载路径，默认为空字符串（当前目录）
                    与课件前面的代码保持一致
    """
    filenames = ['rps.zip', 'rps-test-set.zip']
    for filename in filenames:
        folder_name = filename[:-4]  # 去掉.zip后缀
        folder_path = os.path.join(localfolder, folder_name)
        
        # 检查文件夹是否已存在
        if os.path.exists(folder_path) and os.path.isdir(folder_path):
            print(f'{folder_name}/ 文件夹已存在，跳过下载')
            continue
        
        try:
            os.makedirs(folder_path, exist_ok=True)
            localfile = os.path.join(localfolder, filename)
            
            # TensorFlow官方数据源
            url = 'https://storage.googleapis.com/download.tensorflow.org/data/{}'
            
            if not os.path.exists(localfile):
                print(f'正在下载 {filename}...')
                r = requests.get(url.format(filename), allow_redirects=True)
                open(localfile, 'wb').write(r.content)
                print(f'{filename} 下载完成!')
            
            # 解压ZIP文件到当前目录
            print(f'正在解压 {filename}...')
            with zipfile.ZipFile(localfile, 'r') as zip_ref:
                zip_ref.extractall(localfolder)
            print(f'{filename} 解压完成!')
            
            # 删除ZIP文件（可选，节省空间）
            # os.remove(localfile)
            
        except Exception as e:
            print(f'处理 {filename} 时出错: {e}')


# 执行数据下载（使用默认路径，即当前目录）
download_rps()

# 检查并展示数据集文件夹结构
print("\n数据集文件夹结构:")
print("（ImageFolder要求：子文件夹名即为类别标签）\n")

rps_path = './rps'
if os.path.exists(rps_path):
    for item in sorted(os.listdir(rps_path)):
        item_path = os.path.join(rps_path, item)
        if os.path.isdir(item_path):
            num_files = len([f for f in os.listdir(item_path) if f.endswith(('.png', '.jpg', '.jpeg'))])
            print(f'  rps/{item}/')
            print(f'      └── 包含 {num_files} 张图像')
else:
    print(f'警告: {rps_path} 文件夹不存在！')

rps/ 文件夹已存在，跳过下载
rps-test-set/ 文件夹已存在，跳过下载

数据集文件夹结构:
（ImageFolder要求：子文件夹名即为类别标签）

  rps/paper/
      └── 包含 840 张图像
  rps/rock/
      └── 包含 840 张图像
  rps/scissors/
      └── 包含 840 张图像


#### <font color="#CCFF00">**Step 2：数据标准化与数据增强**</font>

<font color="#FF00A0" >**数据划分（保存索引复用）**</font>：
```
1. 只划分一次：使用random_split获取训练/验证索引
2. 保存索引：train_idx, val_idx供后续复用
3. 计算统计量：使用训练集索引计算mean/std
4. 正式训练：复用相同索引创建Subset，避免seed错误风险导致数据泄露风险
```

<font color="#FF00A0" >**数据标准化**</font>：
- 使用训练集自身的统计信息进行标准化
- 只用训练集计算mean/std，验证集和测试集不可见

<font color="#FF00A0" >**数据增强**</font>：
- **训练时**：随机水平翻转、随机裁剪、颜色抖动等
- **验证/测试时**：仅进行尺寸调整和标准化

In [5]:
# ============================================================================
# Step 2: 数据增强与标准化
# ============================================================================

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch

IMG_SIZE = 128

# 【方案】计算统计量时直接用底层dataset，避免Subset的transform问题
def calculate_mean_std_v2(dataset, indices, batch_size=64, num_workers=2):
    """
    计算指定索引数据的RGB通道均值和标准差
    参数:
        dataset: ImageFolder数据集（有transform属性）
        indices: 要计算的样本索引列表
    """
    temp_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor()
    ])
    
    # 保存原始transform
    original_transform = dataset.transform
    dataset.transform = temp_transform
    
    # 创建Subset用于DataLoader
    subset = torch.utils.data.Subset(dataset, indices)
    
    temp_loader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )
    
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_samples = 0
    
    print("正在计算数据集统计量...")
    for images, _ in temp_loader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(dim=2).sum(dim=0)
        std += images.std(dim=2).sum(dim=0)
        total_samples += batch_samples
    
    mean /= total_samples
    std /= total_samples
    
    # 恢复原始transform
    dataset.transform = original_transform
    
    return mean.tolist(), std.tolist()


# 第一步：加载数据集（临时transform）
temp_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])
full_dataset = ImageFolder(root="./rps", transform=temp_transform)

# 第二步：划分训练/验证索引
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_indices, val_indices = random_split(
    range(len(full_dataset)), [train_size, val_size],
    generator=generator
)

# 保存索引
train_idx = list(train_indices.indices)  # 转为list
val_idx = list(val_indices.indices)

print(f"训练集: {len(train_idx)} 张")
print(f"验证集: {len(val_idx)} 张")

# 第三步：使用自定义函数计算统计量
train_mean, train_std = calculate_mean_std_v2(full_dataset, train_idx)

print(f"\n训练集统计量:")
print(f"均值 (R, G, B): [{train_mean[0]:.4f}, {train_mean[1]:.4f}, {train_mean[2]:.4f}]")
print(f"标准差 (R, G, B): [{train_std[0]:.4f}, {train_std[1]:.4f}, {train_std[2]:.4f}]")

print(f"\nImageNet统计量（供对比）:")
print(f"均值: [0.485, 0.456, 0.406]")
print(f"标准差: [0.229, 0.224, 0.225]")


训练集: 2016 张
验证集: 504 张
正在计算数据集统计量...

训练集统计量:
均值 (R, G, B): [0.8495, 0.8208, 0.8109]
标准差 (R, G, B): [0.2225, 0.2670, 0.2823]

ImageNet统计量（供对比）:
均值: [0.485, 0.456, 0.406]
标准差: [0.229, 0.224, 0.225]


In [6]:
# ============================================================================
# Step 2 (续): 应用数据增强和标准化
# ============================================================================

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import torch

# 定义transform
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std)
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std)
])

# 分别加载训练集和验证集（使用不同的transform）
train_dataset_raw = ImageFolder(root="./rps", transform=train_transform)
val_dataset_raw = ImageFolder(root="./rps", transform=val_transform)

# 使用保存的索引创建Subset
train_dataset = Subset(train_dataset_raw, train_idx)
val_dataset = Subset(val_dataset_raw, val_idx)

# 加载独立的测试集
test_dataset = ImageFolder(root="./rps-test-set", transform=val_transform)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_dataset_raw.classes
print(f"类别映射: {class_names}")
print(f"训练集大小: {len(train_dataset)} 张图像")
print(f"验证集大小: {len(val_dataset)} 张图像")
print(f"测试集大小: {len(test_dataset)} 张图像（独立测试集）")
print(f"批次大小: {BATCH_SIZE}")
print(f"每轮训练迭代次数: {len(train_loader)}")

print(f"\n标准化使用的统计量:")
print(f"均值: {train_mean}")
print(f"标准差: {train_std}")


类别映射: ['paper', 'rock', 'scissors']
训练集大小: 2016 张图像
验证集大小: 504 张图像
测试集大小: 372 张图像（独立测试集）
批次大小: 32
每轮训练迭代次数: 63

标准化使用的统计量:
均值: [0.8495233654975891, 0.8207886219024658, 0.8108795881271362]
标准差: [0.2225153148174286, 0.2670315206050873, 0.2822735905647278]


#### <font color="#CCFF00">**Step 3：网络架构设计**</font>

<font color="#FF00A0" >**深度卷积网络**</font>：

| 组件 | 配置 | 特征图尺寸 |
|:---|:---|:---|
| **卷积块1** | Conv(3→32) + BN + ReLU + MaxPool | 128→64 |
| **卷积块2** | Conv(32→64) + BN + ReLU + MaxPool | 64→32 |
| **卷积块3** | Conv(64→128) + BN + ReLU + MaxPool | 32→16 |
| **卷积块4** | Conv(128→256) + BN + ReLU + MaxPool | 16→8 |
| **全局池化** | AdaptiveAvgPool2d(1) | 8×8→1×1 |
| **分类器** | FC(256→64) + Dropout + FC(64→3) | 极简设计 |


In [7]:
# ============================================================================
# Step 3: 网络架构设计
# ============================================================================

class RPS_CNN(nn.Module):
    """
    RPS图像分类网络 - v12深度版本
    
    设计特点：
    1. 4个卷积块，每个块包含：Conv -> BN -> ReLU -> MaxPool
    2. 逐步增加通道数：32 -> 64 -> 128 -> 256
    3. 每个块后都有池化，快速减小空间尺寸
    4. 极简全连接层：大大减少参数量
    """
    
    def __init__(self, num_classes=3, dropout_rate=0.5):
        super(RPS_CNN, self).__init__()
        
        # 特征提取层 - 4个卷积块
        self.features = nn.Sequential(
            
            # ========== Block 1: 3 -> 32 ==========
            # 输入: 3 x 128 x 128
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 128 -> 64
            
            # ========== Block 2: 32 -> 64 ==========
            # 输入: 32 x 64 x 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 64 -> 32
            
            # ========== Block 3: 64 -> 128 ==========
            # 输入: 64 x 32 x 32
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32 -> 16
            
            # ========== Block 4: 128 -> 256 ==========
            # 输入: 128 x 16 x 16
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16 -> 8
        )
        
        # 全局平均池化 - 进一步减少参数
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # 极简全连接层
        # 输入: 256 (来自global pool)
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, 64),  # 极小中间层
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(64, num_classes)
        )
        
        self._initialize_weights()
    
    def forward(self, x):
        x = self.features(x)      # [B, 256, 8, 8]
        x = self.global_pool(x)   # [B, 256, 1, 1]
        x = x.view(x.size(0), -1) # [B, 256]
        x = self.classifier(x)    # [B, 3]
        return x
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)


# 模型实例化
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'使用设备: {device}')

model = RPS_CNN(num_classes=3, dropout_rate=0.5).to(device)

# 详细参数统计
print("\n" + "=" * 60)
print("网络参数统计")
print("=" * 60)

total_params = 0
conv_params = 0
bn_params = 0
fc_params = 0

for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        p = sum(p.numel() for p in module.parameters())
        conv_params += p
        print(f"Conv2d: {name:20s} - {p:>10,} params")
    elif isinstance(module, nn.BatchNorm2d):
        p = sum(p.numel() for p in module.parameters())
        bn_params += p
    elif isinstance(module, nn.Linear):
        p = sum(p.numel() for p in module.parameters())
        fc_params += p
        print(f"Linear: {name:20s} - {p:>10,} params")

total_params = sum(p.numel() for p in model.parameters())

print("-" * 60)
print(f"卷积层参数量:  {conv_params:>10,} ({conv_params/total_params*100:>5.1f}%)")
print(f"BN层参数量:    {bn_params:>10,} ({bn_params/total_params*100:>5.1f}%)")
print(f"全连接层参数量:{fc_params:>10,} ({fc_params/total_params*100:>5.1f}%)")
print("=" * 60)
print(f"模型总参数量:  {total_params:>10,}")
print("=" * 60)

print("\n【网络设计说明】")
print("- 4个卷积块，每块包含Conv+BN+ReLU+MaxPool")
print("- 使用Global Average Pooling替代Flatten")
print("- 全连接层参数量大幅减少（约2万 vs 420万）")
print("- 更适合小数据集，减少过拟合风险")

# 测试前向传播
test_input = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(device)
test_output = model(test_input)
print(f"\n输入形状: {test_input.shape}")
print(f"输出形状: {test_output.shape}")
print(f"输出示例: {test_output[0].detach().cpu().numpy()}")


使用设备: mps

网络参数统计
Conv2d: features.0           -        896 params
Conv2d: features.4           -     18,496 params
Conv2d: features.8           -     73,856 params
Conv2d: features.12          -    295,168 params
Linear: classifier.1         -     16,448 params
Linear: classifier.4         -        195 params
------------------------------------------------------------
卷积层参数量:     388,416 ( 95.7%)
BN层参数量:           960 (  0.2%)
全连接层参数量:    16,643 (  4.1%)
模型总参数量:     406,019

【网络设计说明】
- 4个卷积块，每块包含Conv+BN+ReLU+MaxPool
- 使用Global Average Pooling替代Flatten
- 全连接层参数量大幅减少（约2万 vs 420万）
- 更适合小数据集，减少过拟合风险

输入形状: torch.Size([4, 3, 128, 128])
输出形状: torch.Size([4, 3])
输出示例: [ 8.026143 37.291924  9.731621]


In [8]:
# ============================================================================
# 网络参数详细统计
# ============================================================================

def count_parameters_detailed(model):
    """
    详细统计网络各层参数数量
    """
    conv_params = 0
    bn_params = 0
    fc_params = 0
    other_params = 0
    
    print("=" * 60)
    print("网络参数统计详情")
    print("=" * 60)
    print(f"{'层类型':<20} {'层名称':<30} {'参数量':<15}")
    print("-" * 60)
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            params = sum(p.numel() for p in module.parameters())
            conv_params += params
            print(f"{'Conv2d':<20} {name:<30} {params:<15,}")
        elif isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
            params = sum(p.numel() for p in module.parameters())
            bn_params += params
            print(f"{'BatchNorm':<20} {name:<30} {params:<15,}")
        elif isinstance(module, nn.Linear):
            params = sum(p.numel() for p in module.parameters())
            fc_params += params
            print(f"{'Linear':<20} {name:<30} {params:<15,}")
    
    total = sum(p.numel() for p in model.parameters())
    
    print("-" * 60)
    print(f"{'卷积层总参数量':<20} {'':<30} {conv_params:<15,} ({conv_params/total*100:.1f}%)")
    print(f"{'BN层总参数量':<20} {'':<30} {bn_params:<15,} ({bn_params/total*100:.1f}%)")
    print(f"{'全连接层总参数量':<20} {'':<30} {fc_params:<15,} ({fc_params/total*100:.1f}%)")
    print("=" * 60)
    print(f"{'模型总参数量':<20} {'':<30} {total:<15,} (100.0%)")
    print("=" * 60)
    
    return {
        'conv': conv_params,
        'bn': bn_params,
        'fc': fc_params,
        'total': total
    }

# 执行参数统计
param_stats = count_parameters_detailed(model)


网络参数统计详情
层类型                  层名称                            参数量            
------------------------------------------------------------
Conv2d               features.0                     896            
BatchNorm            features.1                     64             
Conv2d               features.4                     18,496         
BatchNorm            features.5                     128            
Conv2d               features.8                     73,856         
BatchNorm            features.9                     256            
Conv2d               features.12                    295,168        
BatchNorm            features.13                    512            
Linear               classifier.1                   16,448         
Linear               classifier.4                   195            
------------------------------------------------------------
卷积层总参数量                                             388,416         (95.7%)
BN层总参数量                                      

#### <font color="#CCFF00">**Step 4：优化器配置与学习率调度器**</font>

<font color="#FF00A0" >**优化器选择**</font>：

| 优化器 | 特点 | 适用场景 |
|:---|:---|:---|
| **Adam** | 自适应学习率，收敛快，对超参数不敏感 | **推荐**，大多数情况首选 |
| **SGD + Momentum** | 可能需要更细致的学习率调整，但泛化性能有时更好 | 追求最终精度，配合学习率调度 |

<font color="#FF00A0" >**学习率调度器**</font>：

使用`ReduceLROnPlateau`：
- **作用**：当验证损失不再下降时，自动降低学习率
- **patience**：等待多少个epoch不改善才降低学习率（本例设置为3）
- **factor**：学习率衰减因子（本例为0.5，即每次减半）

In [9]:
# ============================================================================
# Step 4: 优化器配置与学习率调度器
# ============================================================================

# 损失函数
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=5e-4,              
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=1e-4
)

# 学习率调度器
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    threshold=1e-4,
    min_lr=1e-6
)

print('优化器配置完成！')
print(f'优化器类型: {optimizer.__class__.__name__}')
print(f'初始学习率: {optimizer.param_groups[0]["lr"]}')  # 应显示0.0005
print(f'权重衰减: {optimizer.param_groups[0]["weight_decay"]}')


优化器配置完成！
优化器类型: Adam
初始学习率: 0.0005
权重衰减: 0.0001


#### <font color="#CCFF00">**Step 5：早停机制与训练循环**</font>

<font color="#FF00A0" >**早停（Early Stopping）机制**</font>：

```
当验证损失连续 patience 个epoch没有改善时，停止训练
保存验证损失最低时的最佳模型
```

**作用**：防止过拟合，节省训练时间，自动选择最优模型

In [10]:
# ============================================================================
# Step 5: 早停机制与训练循环
# ============================================================================

class EarlyStopping:
    """早停机制"""
    
    def __init__(self, patience=10, min_delta=1e-4, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None
    
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict().copy()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f"早停计数: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            if self.verbose:
                print(f"验证损失改善: {self.best_loss:.4f} -> {val_loss:.4f}")
            self.best_loss = val_loss
            self.best_model_state = model.state_dict().copy()
            self.counter = 0
    
    def save_best_model(self, model, path='rps_best_model.pth'):
        if self.best_model_state is not None:
            torch.save(self.best_model_state, path)
            print(f"最佳模型已保存到 {path}")


def train_epoch(model, loader, criterion, optimizer, device):
    """单个epoch的训练"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    for batch_idx, (inputs, labels) in enumerate(loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc, all_preds, all_labels


def validate(model, loader, criterion, device):
    """验证/测试"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc, all_preds, all_labels


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                device, num_epochs=50, patience=10):
    """完整的模型训练流程"""
    
    early_stopping = EarlyStopping(patience=patience, min_delta=1e-4, verbose=True)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'lr': []
    }
    
    print("=" * 60)
    print("开始训练")
    print("=" * 60)
    
    for epoch in range(num_epochs):
        # 训练
        train_loss, train_acc, train_preds, train_labels = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # 验证
        val_loss, val_acc, val_preds, val_labels = validate(
            model, val_loader, criterion, device
        )
        
        # 学习率调度
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # 记录
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        # 打印进度
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] "
              f"LR: {current_lr:.6f} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
        
        # 每5个epoch打印一次各类别准确率
        if (epoch + 1) % 5 == 0:
            from sklearn.metrics import classification_report
            print("\n验证集分类报告:")
            print(classification_report(val_labels, val_preds, target_names=class_names, digits=4))
        
        # 早停检查
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"\n早停触发！在第 {epoch+1} 轮停止训练")
            print(f"最佳验证损失: {early_stopping.best_loss:.4f}")
            break
    
    print("=" * 60)
    print("训练完成！")
    
    model.load_state_dict(early_stopping.best_model_state)
    early_stopping.save_best_model(model, 'rps_best_model.pth')
    
    return model, history


# 执行训练
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=50,
    patience=10
)


开始训练
Epoch [ 1/50] LR: 0.000500 | Train Loss: 8.7416 Acc: 0.3710 | Val Loss: 2.4006 Acc: 0.4048
Epoch [ 2/50] LR: 0.000500 | Train Loss: 3.0466 Acc: 0.4182 | Val Loss: 0.8812 Acc: 0.4405
验证损失改善: 2.4006 -> 0.8812
Epoch [ 3/50] LR: 0.000500 | Train Loss: 1.8508 Acc: 0.4464 | Val Loss: 0.7363 Acc: 0.8175
验证损失改善: 0.8812 -> 0.7363
Epoch [ 4/50] LR: 0.000500 | Train Loss: 1.3700 Acc: 0.4772 | Val Loss: 0.7829 Acc: 0.5159
早停计数: 1/10
Epoch [ 5/50] LR: 0.000500 | Train Loss: 1.1720 Acc: 0.5005 | Val Loss: 0.7411 Acc: 0.7917

验证集分类报告:
              precision    recall  f1-score   support

       paper     1.0000    0.4631    0.6330       149
        rock     0.9632    0.8626    0.9101       182
    scissors     0.6360    1.0000    0.7775       173

    accuracy                         0.7917       504
   macro avg     0.8664    0.7752    0.7736       504
weighted avg     0.8618    0.7917    0.7827       504

早停计数: 2/10
Epoch [ 6/50] LR: 0.000500 | Train Loss: 1.0737 Acc: 0.5268 | Val Loss: 0.692

In [11]:
# ============================================================================
# 独立测试集评估
# ============================================================================

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

def evaluate_on_test_set(model, test_loader, device):
    """在独立测试集上评估模型性能"""
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


# 加载最佳模型
print("=" * 60)
print("在独立测试集上进行最终评估")
print("=" * 60)

model.load_state_dict(torch.load('rps_best_model.pth'))
model = model.to(device)

test_preds, test_labels, test_probs = evaluate_on_test_set(model, test_loader, device)

# 计算各项指标
test_acc = accuracy_score(test_labels, test_preds)

print(f"\n测试集结果（共 {len(test_labels)} 个样本）:")
print(f"测试准确率: {test_acc:.4f} ({test_acc*100:.2f}%)")

print("\n各类别分类报告:")
print(classification_report(test_labels, test_preds, target_names=class_names, digits=4))

print("\n混淆矩阵:")
cm = confusion_matrix(test_labels, test_preds)
print(cm)

# 检查每个类别的预测情况
print("\n各类别预测统计:")
for i, cls in enumerate(class_names):
    true_count = np.sum(test_labels == i)
    pred_count = np.sum(test_preds == i)
    correct_count = np.sum((test_labels == i) & (test_preds == i))
    print(f"  {cls}: 真实 {true_count:3d} 个, 预测 {pred_count:3d} 个, 正确 {correct_count:3d} 个")

# 计算每个类别的准确率
print("\n各类别准确率:")
for i, cls in enumerate(class_names):
    mask = test_labels == i
    if np.sum(mask) > 0:
        class_acc = accuracy_score(test_labels[mask], test_preds[mask])
        print(f"  {cls}: {class_acc:.4f} ({class_acc*100:.2f}%)")


在独立测试集上进行最终评估

测试集结果（共 372 个样本）:
测试准确率: 0.9462 (94.62%)

各类别分类报告:
              precision    recall  f1-score   support

       paper     1.0000    0.8387    0.9123       124
        rock     0.9185    1.0000    0.9575       124
    scissors     0.9323    1.0000    0.9650       124

    accuracy                         0.9462       372
   macro avg     0.9503    0.9462    0.9449       372
weighted avg     0.9503    0.9462    0.9449       372


混淆矩阵:
[[104  11   9]
 [  0 124   0]
 [  0   0 124]]

各类别预测统计:
  paper: 真实 124 个, 预测 104 个, 正确 104 个
  rock: 真实 124 个, 预测 135 个, 正确 124 个
  scissors: 真实 124 个, 预测 133 个, 正确 124 个

各类别准确率:
  paper: 0.8387 (83.87%)
  rock: 1.0000 (100.00%)
  scissors: 1.0000 (100.00%)


 ### <font color="#FF6B00" size=6>**🤔思考：为什么验证集的Loss和Accuracy在上图中会略好于训练集？**</font>
 ￼

 ### <font color="#FF6B00" size=6>**🤔思考题：**</font>
- 验证集准确率曲线波动大，可能的原因是什么？有必要干预吗？如有必要，如何干预？
- 训练初期，验证集和训练集成绩差距较大，训练后期，二者表现逐渐接近，为什么会有这样的现象，分析原因。
- paper类别的预测成绩略差，可能的原因是？可以采取什么手段改进？

#### <font color="#CCFF00">**完整流程总结**</font>

| 步骤 | 关键技术 | 特点 |
|:---|:---|:---|
| **网络架构** | 4个卷积块 | 每块Conv+BN+ReLU+Pool |
| **特征压缩** | Global Average Pooling | 替代Flatten |
| **分类器** | FC(256→64→3) | 极简设计，仅1.8万参数 |
| **正则化** | Dropout(0.5) | 无BN冲突 |
| **训练** | Adam + ReduceLROnPlateau | 学习率5e-4 |
| **评估** | 独立测试集 | 详细分类报告 |



##  <font color="#FFEA00" >**第2节 现代卷积神经网络**</font>

### **<font color="#39FF14" size=6 >一、CNN架构演进</font>**


  | 网络 | 年份 | 层数 | 参数量 | ImageNet成绩 |作者/机构|特点|
  |:----:|:----:|:----:|:------:|:------------:|:------------:|:------------:|
  | **LeNet-5** | 1998 | 5层 | 60K | 早于ImageNet（始于2009） |Yann LeCun (AT&T Bell Labs)|开创性CNN架构 |
  | **AlexNet** | 2012 | 8层 | 60M | 🏆 2012冠军，错误率: 15.3% |Alex Krizhevsky, Ilya Sutskever, Geoffrey Hinton (University of Toronto)|ReLU+Dropout，GPU训练 |
  | **VGG-16** | 2014 | 16层 | 138M | 🥈 2014亚军，错误率: 7.3% |Karen Simonyan & Andrew Zisserman (牛津大学 Visual Geometry Group)|小卷积核(3x3)堆叠 |
  | <font color="#f6ff00">**GoogLeNet(Inception)**</font> | 2014 | 22层 | 6.8M | 🏆 2014冠军，错误率: 6.7% |Christian Szegedy et al.（Google Research）|Inception模块（多尺度并行），1x1卷积降维|
  | <font color="#f6ff00">**ResNet-50**</font> | 2015 | 50层 | 25.6M | 🏆 2015冠军，错误率: 3.6% |Kaiming He et al. (Microsoft Research Asia)|残差连接，深度可达152+层 |


<font color="#FF00A0" >**关键趋势**</font>：
- 网络深度不断增加
- 卷积核尺寸减小（7x7 → 5x5 → 3x3）
- 引入新的连接方式（残差连接）
- GoogLeNet用22层（比VGG-16的16层更深），但参数量只有6.8M（VGG-16的1/20），却获得了冠军。这说明深度比宽度更重要，启发了后续的ResNet。

### **<font color="#39FF14" size=6 >二、VGG网络</font>**

#### <font color="#CCFF00">**VGG网络架构特点**</font>

VGG（Visual Geometry Group）由牛津大学于2014年提出，其核心思想是：**使用小卷积核（3×3）的堆叠替代大卷积核**。


<font color="#FF00A0" >**核心创新：3×3卷积的堆叠**</font>


<font color="#FF00A0" >**VGG-16详细架构**</font>

```
输入: 224×224×3 (RGB图像)

Block 1 (64通道):
  Conv 3×3, 64 → Conv 3×3, 64 → MaxPool 2×2
  输出: 112×112×64

Block 2 (128通道):
  Conv 3×3, 128 → Conv 3×3, 128 → MaxPool 2×2
  输出: 56×56×128

Block 3 (256通道):
  Conv 3×3, 256 → Conv 3×3, 256 → Conv 3×3, 256 → MaxPool 2×2
  输出: 28×28×256

Block 4 (512通道):
  Conv 3×3, 512 → Conv 3×3, 512 → Conv 3×3, 512 → MaxPool 2×2
  输出: 14×14×512

Block 5 (512通道):
  Conv 3×3, 512 → Conv 3×3, 512 → Conv 3×3, 512 → MaxPool 2×2
  输出: 7×7×512

分类器:
  Flatten → FC 4096 → ReLU → Dropout(0.5)
        → FC 4096 → ReLU → Dropout(0.5)
        → FC 1000 → Softmax
```

<font color="#FF00A0" >**VGG-16参数统计**</font>

| 层 | 参数数量 | 占比 |
|:---|---:|---:|
| 卷积层 | 14.7M | 10.6% |
| 全连接层 | 123.6M | 89.4% |
| **总计** | **138.3M** | **100%** |

<font color="#FF00A0" >**VGG的优缺点**</font>

| 优点 | 缺点 |
|:---|:---|
| 结构简单，易于理解 | 参数量巨大（138M） |
| 小卷积核效率高 | 全连接层占用90%+参数 |
| 特征提取能力强 | 模型体积大（>500MB） |
| 迁移学习效果好 | 推理速度慢 |


In [13]:
# ============================================================================
# VGG网络实现（简化版VGG-16）
# ============================================================================

class VGGBlock(nn.Module):
    """
    VGG基本块：多个3×3卷积 + MaxPool
    """
    def __init__(self, in_channels, out_channels, num_convs):
        super(VGGBlock, self).__init__()
        layers = []
        for i in range(num_convs):
            layers.append(nn.Conv2d(in_channels if i == 0 else out_channels, 
                                   out_channels, kernel_size=3, padding=1))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.block(x)


class VGG16(nn.Module):
    """
    简化版VGG-16（适配128×128输入）
    """
    def __init__(self, num_classes=3):
        super(VGG16, self).__init__()
        
        # 特征提取层
        self.features = nn.Sequential(
            VGGBlock(3, 64, 2),      # 128→64
            VGGBlock(64, 128, 2),    # 64→32
            VGGBlock(128, 256, 3),   # 32→16
            VGGBlock(256, 512, 3),   # 16→8
            VGGBlock(512, 512, 3),   # 8→4
        )
        
        # 分类器
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes)
        )
        
        self._initialize_weights()
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)


# 测试VGG16
vgg_model = VGG16(num_classes=3)
print("=" * 60)
print("VGG-16 Model")
print("=" * 60)

# 统计参数量
total_params = sum(p.numel() for p in vgg_model.parameters())
conv_params = sum(p.numel() for m in vgg_model.modules() if isinstance(m, nn.Conv2d) for p in m.parameters())
fc_params = sum(p.numel() for m in vgg_model.modules() if isinstance(m, nn.Linear) for p in m.parameters())

print(f"Total parameters: {total_params:,}")
print(f"Convolution layers: {conv_params:,} ({conv_params/total_params*100:.1f}%)")
print(f"Fully connected layers: {fc_params:,} ({fc_params/total_params*100:.1f}%)")

# 测试前向传播
test_input = torch.randn(2, 3, 128, 128)
output = vgg_model(test_input)
print(f"\nInput shape: {test_input.shape}")
print(f"Output shape: {output.shape}")


VGG-16 Model
Total parameters: 65,066,819
Convolution layers: 14,714,688 (22.6%)
Fully connected layers: 50,352,131 (77.4%)

Input shape: torch.Size([2, 3, 128, 128])
Output shape: torch.Size([2, 3])


## <font color="#FF6B00" >**🤔思考：为什么总参数量变小了**</font>

- 标准的VGG-16的总参数量是138M，但我们实现的简易版VGG-16的总参数量只有65M
- 为什么少了这么多参数量呢？少在哪里了？

### **<font color="#39FF14" size=6 >三、GoogLeNet（Inception）网络</font>**

<img src="./pics/Inceptionv1.png" alt="替代文字" width="1500" >

<font color="#f6ff00">**ILSVRC2014冠军🏆：GoogLenet/Inception-v1**</font>

<font color="#00c3ff">**GoogLeNet核心创新**</font>


  GoogLeNet（Inception v1）是2014年ImageNet分类竞赛的冠军。与VGG-16相比，它在网络更深（22层）的同时，参数量只有 <font color="#f6ff00">**6.8M**</font>（约为VGG-16的1/
  20），核心创新体现在以下四个方面：

  <font color="#f6ff00"> **1. Inception 模块：多尺度特征并行提取**</font>

  <font color="#f6ff00">**2. $1\times1$ 卷积：降维（Bottleneck）**</font>

  <font color="#f6ff00">**3. 全局平均池化（Global Average Pooling）替代全连接层**</font>

  <font color="#f6ff00">**4. 辅助分类器（Auxiliary Classifiers）**</font>


  ---

  <font color="#FF00A0"> **一句话总结** </font>
  > GoogLeNet 证明了：**通过精巧的架构设计（Inception 模块 + $1\times1$ 降维 + 全局平均池化），可以在网络更深、表达能力更强的同时，将参数量压缩到极小。** 这一思想直接启发了后续 ResNet、MobileNet 等高效网络的设计。



#### <font color="#CCFF00">**Inception模块**</font>

<img src="./pics/inception模块v1.png" alt="替代文字" width="900" >

Inception模块包含四个并行分支：
1. **1×1卷积分支**：降维 + 特征提取
2. **3×3卷积分支**：中等感受野
3. **5×5卷积分支**：大感受野
4. **Max Pooling分支**：保留显著特征

**关键创新**：使用1×1卷积进行**维度降维**，在增加深度和宽度的同时控制计算量。


##### <font color="#00ffbf">**Inception-v1模块详细架构**</font>

```
输入：192通道 × 28×28

分支1：1×1 Conv (64核) ─────────────→ 64通道 × 28×28
分支2：1×1 Conv (96核) → 3×3 Conv (128核) → 128通道 × 28×28
分支3：1×1 Conv (16核) → 5×5 Conv (32核) → 32通道 × 28×28
分支4：MaxPool(3×3) → 1×1 Conv (32核) → 32通道 × 28×28

输出拼接：64 + 128 + 32 + 32 = 256通道 × 28×28
```
##### <font color="#00ffbf">**Inception-v2完整架构**</font>

Szegedy et al., 2016

##### <font color="#00ffbf">**Inception-v3的改进**</font>

<font color="#f6ff00">**在 Inception-v2 上添加的额外技术**</font>

  <font color="#FF00A0">**1. RMSProp 优化器**</font>
  使用自适应学习率的 RMSProp 优化器替代传统的 SGD，加速深层网络的收敛。

  <font color="#FF00A0">**2. 在辅助分类器中添加批归一化（BatchNorm）**</font>
  在 GoogLeNet 原有辅助分类器（Auxiliary Classifiers）的基础上添加 Batch Normalization，进一步缓解梯度消失问题并提升训练稳定性。

  <font color="#FF00A0">**3. 标签平滑（Label Smoothing）**</font>
  一种防止过拟合的正则化技术。它将原始的 hard target（one-hot 标签）替换为 soft target，避免模型对训练样本过度自信。

  ---

  **原始 one-hot 标签分布：**

  $$
  P_i =
  \begin{cases}
  1, & \text{if } (i = y) \\[6pt]
  0, & \text{if } (i \neq y)
  \end{cases}
  $$

  **经过 Label Smoothing 后的标签分布：**

  $$
  P_i =
  \begin{cases}
  (1 - \varepsilon), & \text{if } (i = y) \\[12pt]
  \dfrac{\varepsilon}{K - 1}, & \text{if } (i \neq y)
  \end{cases}
  $$

  其中：
  - $y$ 为真实类别
  - $K$ 为类别总数
  - $\varepsilon$ 为平滑因子（通常取 $0.1$）

  ---

  **对应的损失函数变化：**

  原始的交叉熵损失：

  $$
  Loss = -\sum_{i=1}^{K} p_i \log q_i
  $$

  使用 Label Smoothing 后，将平滑后的概率 $p_i$ 代入交叉熵公式，等价于对损失进行了如下权重调整：

  $$
  Loss_i =
  \begin{cases}
  (1 - \varepsilon) \cdot Loss, & \text{if } (i = y) \\[8pt]
  \varepsilon \cdot Loss, & \text{if } (i \neq y)
  \end{cases}
  $$

  > **物理意义**：标签平滑通过降低正确标签的置信度（从 1 降到 $1-\varepsilon$），并给错误标签分配微小的概率（$\frac{\varepsilon}{K-1}$）
  ，阻止模型将 logits 推得过大，从而起到正则化作用。

#### <font color="#CCFF00">**1×1卷积详解**</font>

##### <font color="#00ffbf">**1×1卷积的核心作用**</font>

1×1卷积（Point-wise Convolution）是深度学习中非常精巧的设计，看似简单却功能强大。

**定义**：卷积核尺寸为 1×1 的卷积操作，只在**通道维度**上做线性组合，不改变空间尺寸。

```
输入特征图: [C_in, H, W]
           ↓
    1×1 卷积核: [C_out, C_in, 1, 1]
           ↓
输出特征图: [C_out, H, W]  (H×W 不变，通道数变为 C_out)
```

<font color="#FF00A0" >**关键特性**</font>：
- 感受野 = 1×1（只看当前位置）
- 不融合空间信息（不邻域）
- **只融合通道信息**

##### <font color="#00ffbf">**作用一：降维/升维（Channel Transformation）**</font>

**最常用功能**：灵活调整通道数，控制计算量。

```
输入: 256通道 ──→ 1×1卷积(64核) ──→ 输出: 64通道  (降维，减少75%计算量)
输入: 64通道  ──→ 1×1卷积(256核) ──→ 输出: 256通道 (升维，恢复表达能力)
```

<font color="#FF00A0" >**关键洞察**</font>：如果没有 1×1 卷积降维，直接做 3×3 和 5×5 卷积，计算量会爆炸。1×1卷积就像\"交通枢纽\"，先压缩信息，再分发到不同处理分支。

##### <font color="#00ffbf">**作用二：跨通道信息融合**</font>

**抽象解释**：每个输出通道是**所有输入通道的加权组合**。

**作用**：让网络自动学习如何\"混合\"不同通道的特征（如边缘+颜色=特定物体）。

> 💡 **类比**：1×1卷积就像是给每个像素位置安排了一个\"小型全连接层\"，这个全连接层的输入是所有通道在该位置的值，输出是新通道在该位置的值。

##### <font color="#00ffbf">**1×1卷积与普通卷积对比**</font>

| 特性 | 1×1 卷积 | 3×3 卷积 |
|:----:|:--------:|:--------:|
| 感受野 | 1×1 | 3×3 |
| 空间信息 | 不融合 | 融合邻域 |
| 通道信息 | 完全融合 | 局部融合 |
| 计算量 | 小 | 大 |
| 主要用途 | 调整通道数、跨通道交互 | 提取空间特征 |
| 参数量 | C_in × C_out | C_in × C_out × 9 |

<font color="#FF00A0" >**一句话记忆**</font>：1×1 卷积是深度网络中的\"交通警察\"——不负责空间侦查（那是3×3的活），专门指挥不同通道之间的信息流动和重组。

##### <font color="#00ffbf">**计算效率对比**</font>


**场景**：输入 256×56×56，输出 256×56×56

| 操作 | 计算量 (FLOPs) | 参数量 |
|:----:|:--------------:|:------:|
| 直接 3×3 conv | 256×256×3×3×56×56 = **1.85G** | 256×256×3×3 = 589K |
| 1×1→3×3→1×1 (Bottleneck) | 256×64 + 64×64×3×3 + 64×256 = **0.17G** | 16K + 36K + 16K = 68K |
| **节省** | **90%** | **88%** |

<font color="#FF00A0" >**结论**</font>：这就是 ResNet-50 比 VGG-16 深但计算量更少的秘密！

#### <font color="#CCFF00">**全局平均池化（Global Average Pooling, GAP）**</font>

  全局平均池化是将每个特征图（channel）上所有空间位置的数值取平均，从而把 $H \times W \times C$ 的三维张量压缩为 $1 \times 1 \times C$
  的向量。

  **示例**：若最后一层卷积输出为 $7 \times 7 \times 512$，GAP 会对这 512 个通道分别求平均，最终得到一个长度为 512 的特征向量。

  ---

  ##### <font color="#00ffbf">**为什么 GAP 能代替全连接层？**</font>


  ##### <font color="#00ffbf">**效果不会变差吗？为什么准确率反而有保障？**</font>


  <font color="#FF00A0">**一句话总结**</font>

  > **全局平均池化不是"将就"地替代全连接层，而是一种更优雅的总结方式**：它假设深度卷积的每个通道已经是一个合格的"语义检测器"，只需统计它们的平均活跃度即可分类。这既消灭了上亿参数，又通过结构约束增强了特征的泛化能力，所以 GoogLeNet 能在参数量仅为 VGG 1/20 的情况下取得更好的效果。

#### <font color="#CCFF00">**辅助分类器（Auxiliary Classifiers）**</font>

<font color="#FF00A0" >**辅助分类器在网络中的位置**</font>

- **Aux-1**：接在Inception(4a)之前（网络深度的1/3处）
- **Aux-2**：接在Inception(4d)之后（网络深度的2/3处）

辅助分类器作用：
1. **缓解梯度消失**：为浅层提供额外的梯度信号
2. **正则化效果**：防止过拟合
3. **训练时使用，推理时丢弃**


<font color="#FF00A0" >**1. 缓解梯度消失**</font>

```
总损失 = 主分类器损失 + 0.3 × 辅助分类器1损失 + 0.3 × 辅助分类器2损失
```

> **一句话总结**：辅助分类器把一条"长距离回传"的梯度路径，拆成了多条"短距离回传"的路径，让浅层网络也能获得足够强的梯度信号。

<font color="#FF00A0" >**2. 如何起到正则化效果？**</font>

  辅助分类器就是这个道理：网络不能只训练"深层特征"来讨好最终输出，它在中间层就必须表现好。这迫使网络学习**更通用、更鲁棒的中间表示**。


<font color="#FF00A0" >**3. 训练 vs 推理的差异处理**</font>

**训练时**：三个分类器都计算损失，加权求和后统一回传梯度。

**推理时**：完全丢弃辅助分类器，只用主分类器，节省计算量，避免干扰




#### <font color="#CCFF00">**GoogLeNet与其他网络的参数对比**</font>

| 网络 | 参数量 | 准确率 | 优势 |
|:---|---:|:---|:---|
| VGG-16 | 138M | 71.5% | 结构简单 |
| **GoogLeNet** | **6.8M** | **69.8%** | **参数少20倍** |
| ResNet-50 | 25.6M | 76.0% | 残差连接 |

GoogLeNet用1/20的参数达到了接近VGG的准确率！


In [15]:
# ============================================================================
# GoogLeNet Inception模块实现（完整版）
# ============================================================================

class InceptionModule(nn.Module):
    """
    经典Inception模块：并行多尺度特征提取
    注意：原始GoogLeNet论文中使用AvgPool，无BatchNorm
    """
    def __init__(self, in_channels, ch1x1, ch3x3red, ch3x3, ch5x5red, ch5x5, pool_proj):
        super(InceptionModule, self).__init__()

        # 分支1：1×1卷积
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, kernel_size=1),
            nn.ReLU(inplace=True)
        )

        # 分支2：1×1降维 → 3×3卷积
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3red, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch3x3red, ch3x3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        # 分支3：1×1降维 → 5×5卷积
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5red, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch5x5red, ch5x5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )

        # 分支4：平均池化 → 1×1投影（原始论文使用AvgPool）
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)

        # 通道拼接
        return torch.cat([b1, b2, b3, b4], dim=1)


class AuxiliaryClassifier(nn.Module):
    """
    辅助分类器：用于训练阶段缓解梯度消失并提供正则化
    推理阶段丢弃不使用
    """
    def __init__(self, in_channels, num_classes):
        super(AuxiliaryClassifier, self).__init__()
        self.classifier = nn.Sequential(
            nn.AvgPool2d(kernel_size=5, stride=3),
            nn.Conv2d(in_channels, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 2 * 2, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.7),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)


class SimpleGoogLeNet(nn.Module):
    """
    简化版GoogLeNet（适配128×128输入）
    包含：初始卷积层、Inception模块、辅助分类器、全局平均池化
    """
    def __init__(self, num_classes=3, use_aux=True):
        super(SimpleGoogLeNet, self).__init__()
        self.use_aux = use_aux

        # 初始卷积
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )  # 128→32

        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )  # 32→16

        # Inception 3
        self.inception3a = InceptionModule(192, 64, 96, 128, 16, 32, 32)   # 192→256
        self.inception3b = InceptionModule(256, 128, 128, 192, 32, 96, 64) # 256→480
        self.maxpool3 = nn.MaxPool2d(3, stride=2, padding=1)  # 16→8

        # Inception 4
        self.inception4a = InceptionModule(480, 192, 96, 208, 16, 48, 64)  # 480→512
        self.inception4b = InceptionModule(512, 160, 112, 224, 24, 64, 64) # 512→512
        self.inception4c = InceptionModule(512, 128, 128, 256, 24, 64, 64)  # 512→512
        self.inception4d = InceptionModule(512, 112, 144, 288, 32, 64, 64)  # 512→528
        self.inception4e = InceptionModule(528, 256, 160, 320, 32, 128, 128) # 528→832
        self.maxpool4 = nn.MaxPool2d(3, stride=2, padding=1)  # 8→4

        # 辅助分类器1（接在inception4a之后，8×8特征图）
        if self.use_aux:
            self.aux1 = AuxiliaryClassifier(512, num_classes)
            # 辅助分类器2（接在inception4d之后，8×8特征图）
            self.aux2 = AuxiliaryClassifier(528, num_classes)

        # Inception 5
        self.inception5a = InceptionModule(832, 256, 160, 320, 32, 128, 128) # 832→832
        self.inception5b = InceptionModule(832, 384, 192, 384, 48, 128, 128) # 832→1024

        # 全局平均池化 + 分类器
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)

        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)

        x = self.inception4a(x)
        aux1 = self.aux1(x) if self.training and self.use_aux else None

        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        aux2 = self.aux2(x) if self.training and self.use_aux else None

        x = self.inception4e(x)
        x = self.maxpool4(x)

        x = self.inception5a(x)
        x = self.inception5b(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)

        # 训练时返回主输出+两个辅助输出；推理时只返回主输出
        if self.training and self.use_aux:
            return x, aux1, aux2
        return x


# ========================= 测试代码 =========================
if __name__ == "__main__":
    print("=" * 60)
    print("Inception Module Test")
    print("=" * 60)

    inception_test = InceptionModule(
        in_channels=192, ch1x1=64, ch3x3red=96, ch3x3=128,
        ch5x5red=16, ch5x5=32, pool_proj=32
    )
    test_input = torch.randn(2, 192, 28, 28)
    test_output = inception_test(test_input)
    print(f"Input shape: {test_input.shape}")
    print(f"Output shape: {test_output.shape}")
    print(f"Channel calculation: 64+128+32+32 = {64+128+32+32}")

    print("\n" + "=" * 60)
    print("SimpleGoogLeNet Model")
    print("=" * 60)

    # 推理模式（关闭辅助分类器输出）
    googlenet_eval = SimpleGoogLeNet(num_classes=3, use_aux=True)
    googlenet_eval.eval()

    total_params = sum(p.numel() for p in googlenet_eval.parameters())
    print(f"Total parameters: {total_params:,}")

    test_input = torch.randn(2, 3, 128, 128)
    with torch.no_grad():
        test_output = googlenet_eval(test_input)
    print(f"\n[Eval] Input shape: {test_input.shape}")
    print(f"[Eval] Output shape: {test_output.shape}")

    # 训练模式（返回主输出+aux1+aux2）
    googlenet_train = SimpleGoogLeNet(num_classes=3, use_aux=True)
    googlenet_train.train()
    output_main, output_aux1, output_aux2 = googlenet_train(test_input)
    print(f"\n[Train] Main output: {output_main.shape}")
    print(f"[Train] Aux1 output: {output_aux1.shape}")
    print(f"[Train] Aux2 output: {output_aux2.shape}")

    # 计算总损失示例
    criterion = nn.CrossEntropyLoss()
    labels = torch.randint(0, 3, (2,))
    loss_main = criterion(output_main, labels)
    loss_aux1 = criterion(output_aux1, labels)
    loss_aux2 = criterion(output_aux2, labels)
    total_loss = loss_main + 0.3 * loss_aux1 + 0.3 * loss_aux2
    print(f"\nLoss_main: {loss_main.item():.4f}")
    print(f"Loss_aux1: {loss_aux1.item():.4f}")
    print(f"Loss_aux2: {loss_aux2.item():.4f}")
    print(f"Total loss: {total_loss.item():.4f}")



Inception Module Test
Input shape: torch.Size([2, 192, 28, 28])
Output shape: torch.Size([2, 256, 28, 28])
Channel calculation: 64+128+32+32 = 256

SimpleGoogLeNet Model
Total parameters: 7,162,617

[Eval] Input shape: torch.Size([2, 3, 128, 128])
[Eval] Output shape: torch.Size([2, 3])

[Train] Main output: torch.Size([2, 3])
[Train] Aux1 output: torch.Size([2, 3])
[Train] Aux2 output: torch.Size([2, 3])

Loss_main: 31.1747
Loss_aux1: 1.1069
Loss_aux2: 2.1131
Total loss: 32.1407


### **<font color="#39FF14" size=6 >四、ResNet（残差网络）</font>**

#### <font color="#CCFF00">**ResNet解决的问题：网络退化**</font>

在ResNet之前，研究人员发现：**当网络层数增加到一定程度后，准确率不但不提升，反而下降**。

He et al.，《Deep Residual Learning for Image Recognition》（CVPR 2016）

#### <font color="#CCFF00">**网络退化 vs ResNet 的解决思路**</font>

  > **结果**：信息不会被迫经过多层无意义的扭曲，深层网络至少不会比浅层差，而且多出来的层可以逐步学习更精细的修正。


#### <font color="#CCFF00">**ResNet残差连接效果**</font>


##### <font color="#00ffbf">**残差连接为什么有效？**</font>

1. **梯度高速公路**：反向传播时梯度可以直接回传，缓解梯度消失
2. **恒等映射学习**：如果某层无用，网络可以学习让F(x)≈0
3. **集成学习视角**：残差网络可以看作多个浅层网络的集成

<font color="#f6ff00">**ResNet 最精妙的地方之一**</font>：结构上极其简单（就加了个 $+x$），却在数学上等价于一个隐式的深度集成模型。所以 152 层的 ResNet 既不会退化，又比同等深度的普通网络更鲁棒。

#### <font color="#FF6B00" > **附加思考🤔：为什么残差网络可以看作是多个浅层网络的集成** </font>
- 提示：关键点——理解“多通路的参数共享“
- 《Residual Networks Behave Like Ensembles of Relatively Shallow Networks》，2016

In [18]:
# ============================================================================
# ResNet残差网络实现
# ============================================================================

class BasicBlock(nn.Module):
    """
    基础残差块（用于ResNet-18/34）
    """
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = downsample  # 用于匹配尺寸
        self.stride = stride

    def forward(self, x):
        identity = x  # 保存输入用于残差连接

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 如果尺寸不匹配，使用downsample
        if self.downsample is not None:
            identity = self.downsample(x)

        # 残差连接
        out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
      """
      Bottleneck残差块（用于ResNet-50/101/152）
      """
      expansion = 4  # 输出通道是输入的4倍

      def __init__(self, in_channels, out_channels, stride=1, downsample=None):
          super(Bottleneck, self).__init__()

          # 1×1降维
          self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
          self.bn1 = nn.BatchNorm2d(out_channels)

          # 3×3卷积
          self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                  stride=stride, padding=1, bias=False)
          self.bn2 = nn.BatchNorm2d(out_channels)

          # 1×1升维
          self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion,
                                  kernel_size=1, bias=False)
          self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

          self.relu = nn.ReLU(inplace=True)
          self.downsample = downsample
          self.stride = stride

      def forward(self, x):
          identity = x

          out = self.conv1(x)
          out = self.bn1(out)
          out = self.relu(out)

          out = self.conv2(out)
          out = self.bn2(out)
          out = self.relu(out)

          out = self.conv3(out)
          out = self.bn3(out)

          if self.downsample is not None:
              identity = self.downsample(x)

          out += identity  # ✅ 现在通道数匹配了
          out = self.relu(out)

          return out


class SimpleResNet(nn.Module):
    """
    简化版ResNet（适配128×128输入）
    """
    def __init__(self, block, layers, num_classes=3):
        super(SimpleResNet, self).__init__()

        self.in_channels = 64

        # 初始卷积
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )  # 128→32

        # 残差层
        self.layer1 = self._make_layer(block, 64, layers[0])   # 32×32
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)  # 16×16
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)  # 8×8
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)  # 4×4

        # 全局平均池化
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        # 权重初始化
        self._initialize_weights()

    def _make_layer(self, block, out_channels, num_blocks, stride=1):
        downsample = None

        # 当stride≠1或通道数变化时，需要downsample
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                        kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))

        self.in_channels = out_channels * block.expansion

        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)


def create_resnet18(num_classes=3):
    """创建ResNet-18"""
    return SimpleResNet(BasicBlock, [2, 2, 2, 2], num_classes)


def create_resnet34(num_classes=3):
    """创建ResNet-34"""
    return SimpleResNet(BasicBlock, [3, 4, 6, 3], num_classes)


def create_resnet50(num_classes=3):
    """创建ResNet-50"""
    return SimpleResNet(Bottleneck, [3, 4, 6, 3], num_classes)


# 测试代码
if __name__ == "__main__":
    print("=" * 60)
    print("Residual Block Test")
    print("=" * 60)

    # 测试BasicBlock
    basic_block = BasicBlock(in_channels=64, out_channels=64)
    test_input = torch.randn(2, 64, 56, 56)
    test_output = basic_block(test_input)
    print(f"BasicBlock:")
    print(f"  Input: {test_input.shape}")
    print(f"  Output: {test_output.shape}")
    print(f"  Has residual connection: {test_output.shape == test_input.shape}")

    # 测试带降采样的BasicBlock
    downsample = nn.Sequential(
        nn.Conv2d(64, 128, kernel_size=1, stride=2, bias=False),
        nn.BatchNorm2d(128)
    )
    basic_block_down = BasicBlock(in_channels=64, out_channels=128, stride=2, downsample=downsample)
    test_output_down = basic_block_down(test_input)
    print(f"\nBasicBlock with downsampling:")
    print(f"  Input: {test_input.shape}")
    print(f"  Output: {test_output_down.shape}")

    # 测试Bottleneck
    print("\n" + "-" * 40)

    # 添加 downsample，将 64 通道映射到 256 通道
    downsample_bt = nn.Sequential(
      nn.Conv2d(64, 256, kernel_size=1, bias=False),
      nn.BatchNorm2d(256)
  )
    bottleneck = Bottleneck(in_channels=64, out_channels=64, downsample=downsample_bt)
    test_input_bt = torch.randn(2, 64, 56, 56)
    test_output_bt = bottleneck(test_input_bt)
    print(f"Bottleneck:")
    print(f"  Input: {test_input_bt.shape}")
    print(f"  Output: {test_output_bt.shape}")
    print(f"  Channel expansion: 64 -> {64 * Bottleneck.expansion}")

    # 测试带降采样的Bottleneck
    downsample_bt_down = nn.Sequential(
        nn.Conv2d(64, 256, kernel_size=1, stride=2, bias=False),
        nn.BatchNorm2d(256)
    )
    bottleneck_down = Bottleneck(in_channels=64, out_channels=64, stride=2, downsample=downsample_bt_down)
    test_output_bt_down = bottleneck_down(test_input_bt)
    print(f"\nBottleneck with downsampling:")
    print(f"  Input: {test_input_bt.shape}")
    print(f"  Output: {test_output_bt_down.shape}")

    # 测试完整ResNet-18
    print("\n" + "=" * 60)
    print("ResNet-18 Model")
    print("=" * 60)

    resnet18 = create_resnet18(num_classes=3)
    total_params = sum(p.numel() for p in resnet18.parameters())
    print(f"Total parameters: {total_params:,}")

    test_input = torch.randn(2, 3, 128, 128)
    test_output = resnet18(test_input)
    print(f"\nInput shape: {test_input.shape}")
    print(f"Output shape: {test_output.shape}")

    # 测试完整ResNet-50
    print("\n" + "=" * 60)
    print("ResNet-50 Model")
    print("=" * 60)

    resnet50 = create_resnet50(num_classes=3)
    total_params_50 = sum(p.numel() for p in resnet50.parameters())
    print(f"Total parameters: {total_params_50:,}")

    test_output_50 = resnet50(test_input)
    print(f"\nInput shape: {test_input.shape}")
    print(f"Output shape: {test_output_50.shape}")



Residual Block Test
BasicBlock:
  Input: torch.Size([2, 64, 56, 56])
  Output: torch.Size([2, 64, 56, 56])
  Has residual connection: True

BasicBlock with downsampling:
  Input: torch.Size([2, 64, 56, 56])
  Output: torch.Size([2, 128, 28, 28])

----------------------------------------
Bottleneck:
  Input: torch.Size([2, 64, 56, 56])
  Output: torch.Size([2, 256, 56, 56])
  Channel expansion: 64 -> 256

Bottleneck with downsampling:
  Input: torch.Size([2, 64, 56, 56])
  Output: torch.Size([2, 256, 28, 28])

ResNet-18 Model
Total parameters: 11,178,051

Input shape: torch.Size([2, 3, 128, 128])
Output shape: torch.Size([2, 3])

ResNet-50 Model
Total parameters: 23,514,179

Input shape: torch.Size([2, 3, 128, 128])
Output shape: torch.Size([2, 3])


#### <font color="#CCFF00">**ResNet网络性能对比**</font>


<font color="#f6ff00" size=6 >**1000层的ResNet也成功运用在CIFAR-10数据集上**</font>

### **<font color="#39FF14" size=6 >五、CNN架构的参数和性能对比</font>**


#### <font color="#CCFF00">**CNN 架构对比：准确率 vs 参数量 vs 计算量**</font>


  <font color="#f6ff00">**核心结论**</font>：更好的网络结构设计（ResNet、Inception）可以用<font color="#f6ff00">**更少的参数和计算量**</font>，达到比"暴力堆叠"（VGG）更高的准确率。

  | 模型 | 参数量 | 计算量 | Top-1 准确率 | 核心特点 |
  |:---|:---|:---|:---|:---|
  | **AlexNet** | ~60M | ~0.7 G-Ops | ~57% | 开山之作，结构简单 |
  | **VGG-16/19** | **~138M**（最大） | **~30+ G-Ops** | ~70% | 堆叠深而宽，参数冗余严重 |
  | **GoogLeNet** | ~6.8M（极小） | ~3 G-Ops | ~68.5% | Inception + 全局平均池化 |
  | **ResNet-18/34** | ~11M / ~21M | ~2-4 G-Ops | ~70% / ~73% | 残差连接，解决退化 |
  | **ResNet-50/101/152** | ~25M / ~44M / ~60M | ~4-11 G-Ops | **~76-78%** | 越深越准，参数效率极高 |
  | **Inception-v3/v4** | ~23M / ~42M | ~5-12 G-Ops | ~78-80% | 多尺度 + 残差融合 |

  **关键洞察**：
  - **VGG** 靠**堆参数和计算量**换性能，效率最低。
  - **GoogLeNet** 用精巧设计（$1\times1$ 降维、全局平均池化）大幅压缩了参数。
  - **ResNet** 和 **Inception** 通过**残差连接 / 多尺度设计**，实现了"**更少参数、更少计算、更高准确率**"的跃迁。ResNet-152 是最极端的例子——它比 VGG-16 **更深、更准确**，但**模型反而更轻量**。

  ---

  #### <font color="#CCFF00">**迁移学习推荐：首选 ResNet-50**</font>

  在**通用迁移学习**这个赛道上，ResNet-50 的"泛化能力 + 推理速度 + 生态支持"通常是更稳妥的选择。

  | 场景 | 推荐模型 | 理由 |
  |:---|:---|:---|
  | **通用迁移学习首选** | **ResNet-50** | 平衡了准确率、速度、泛化性和生态支持 |
  | 追求极致精度 | ResNet-101/152 或 EfficientNet | 在 ResNet 基础上升级 |
  | 移动端/嵌入式 | MobileNet、ShuffleNet | 专门为低算力设计 |
  | 特定分类任务且数据量大 | Inception-v3/v4 | 可作为备选尝试 |

  **建议**：
  - **算力充足、追求极致精度** → 升级到 **ResNet-101/152** 或 **EfficientNet**
  - **算力受限、移动/边缘设备** → 选择 **ResNet-18/34** 或 **MobileNet**




 

##  <font color="#FFEA00" >**第3节 迁移学习（Transfer Learning）**</font>

### **<font color="#39FF14" size=6 >迁移学习核心概念</font>**

#### <font color="#CCFF00">**什么是迁移学习？**</font>

迁移学习（Transfer Learning）是一种利用已有知识解决新问题的方法：“站在巨人的肩膀上“

<font color="#FF00A0">**核心思想**</font>：
1. 一些大型科技公司使用海量数据和算力训练出强大的模型
2. 训练完成后，发布模型的架构和预训练权重
3. 我们可以将这些预训练权重作为起点，针对自己的任务进行微调

**迁移学习的优势**：
- 无需从头训练大型模型（节省时间和计算资源）
- 在数据量较少时也能获得良好效果
- 利用预训练模型学到的通用特征

#### <font color="#CCFF00">**ImageNet与预训练模型**</font>

**ImageNet**是一个大型图像数据库，包含超过1400万张图像，涵盖2万多个类别。

**ImageNet Large Scale Visual Recognition Challenge (ILSVRC)**：
- 2010-2017年举办的图像分类竞赛
- 催生了AlexNet、VGG、Inception、ResNet等经典架构


| 年份 | 架构 | 主要贡献 |
|:----:|:----:|:---------|
| 2012 | AlexNet | 8层，ReLU+Dropout，GPU训练 |
| 2014 | VGG | 16-19层，小卷积核(3×3)堆叠 |
| 2014 | Inception | 并行多尺度卷积，1×1降维 |
| 2015 | ResNet | 残差连接，可训练152+层 |

### **<font color="#39FF14" size=6 >预训练模型的下载及权重加载</font>**

#### <font color="#CCFF00">**加载预训练权重的方法**</font>

**方法一：直接加载（推荐）**
```python
from torchvision.models import alexnet
model = alexnet(pretrained=True)  # 自动下载并加载预训练权重
```

**方法二：从URL下载（更灵活）**
```python
from torchvision.models.alexnet import model_urls
from torchvision.models.utils import load_state_dict_from_url

# 获取权重URL
url = model_urls['alexnet']
print(url)  # https://download.pytorch.org/models/alexnet-owt-4df8aa71.pth

# 下载权重
state_dict = load_state_dict_from_url(url, model_dir='pretrained', progress=True)

# 加载到模型
model = alexnet(pretrained=False)
model.load_state_dict(state_dict)
```

In [19]:
# 预训练模型加载示例
from torchvision.models import alexnet, resnet18, inception_v3

# 加载AlexNet（带预训练权重）
alex = alexnet(pretrained=True)
print("AlexNet加载成功！")
print(f"分类器最后一层: {alex.classifier[6]}")

AlexNet加载成功！
分类器最后一层: Linear(in_features=4096, out_features=1000, bias=True)


### **<font color="#39FF14" size=6 >模型冻结与微调</font>**



- 冻结部分的网络已经在 ImageNet 上预训练好了，学到了通用的边缘、纹理、形状等特征。对于新任务（比如识别猫狗、医疗影像），这些底层特征往往是通用的，不需要重新学。
- 冻结预训练网络的特征提取部分，只训练（或替换后训练）顶部的分类器，让模型快速适应自己的数据集。

#### <font color="#CCFF00">**模型冻结（Model Freezing）**</font>

**概念**：冻结模型意味着模型的参数在训练过程中不会被更新。

**为什么需要冻结？**
- 预训练模型已经学到了通用的特征（边缘、纹理、形状等）
- 我们只需要训练最后的分类层来适应特定任务
- 避免在少量数据上过度调整预训练权重，导致过拟合

**冻结实现**：通过设置`requires_grad = False`

In [20]:
# 模型冻结实现
def freeze_model(model):
    """冻结模型的所有参数"""
    for param in model.parameters():
        param.requires_grad = False

def unfreeze_model(model):
    """解冻模型的所有参数"""
    for param in model.parameters():
        param.requires_grad = True

# 示例：冻结AlexNet
alex = alexnet(pretrained=True)
freeze_model(alex)

# 检查参数是否冻结
trainable_params = sum(p.numel() for p in alex.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in alex.parameters())

print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")
print(f"已冻结参数量: {total_params - trainable_params:,}")

总参数量: 61,100,840
可训练参数量: 0
已冻结参数量: 61,100,840


#### <font color="#CCFF00">**替换顶层分类器**</font>

冻结模型后，需要替换最后的分类层以适应新任务：

In [21]:
# 替换AlexNet的顶层分类器（3分类示例：石头剪刀布）
import torch.nn as nn

# 假设我们的任务有3个类别
num_classes = 3

# 替换最后一层
alex.classifier[6] = nn.Linear(4096, num_classes)

# 检查哪些参数需要训练
print("需要训练的参数:")
for name, param in alex.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

需要训练的参数:
  classifier.6.weight: torch.Size([3, 4096])
  classifier.6.bias: torch.Size([3])


#### <font color="#CCFF00">**预训练模型构成及迁移学习方案替换层对照表**</font>

使用预训练模型时，需要替换最后一层（分类层）以适应自己的任务：

  | 模型 | 输入尺寸 | 分类器层 | 替换方式 | 备注 |
  |:----:|:--------:|:---------|:---------|:-----|
  | **AlexNet** | 224×224 | `model.classifier[6]` | `nn.Linear(4096, num_classes)` | — |
  | **VGG** | 224×224 | `model.classifier[6]` | `nn.Linear(4096, num_classes)` | 适用于 VGG16 / VGG19 |
  | **ResNet-18/34** | 224×224 | `model.fc` | `nn.Linear(512, num_classes)` | — |
  | **ResNet-50/101/152** | 224×224 | `model.fc` | `nn.Linear(2048, num_classes)` | — |
  | **DenseNet-121** | 224×224 | `model.classifier` | `nn.Linear(1024, num_classes)` | — |
  | **DenseNet-169** | 224×224 | `model.classifier` | `nn.Linear(1664, num_classes)` | — |
  | **DenseNet-201** | 224×224 | `model.classifier` | `nn.Linear(1920, num_classes)` | — |
  | **DenseNet-161** | 224×224 | `model.classifier` | `nn.Linear(2208, num_classes)` | — |
  | **InceptionV3** | 299×299 | `model.fc` | `nn.Linear(2048, num_classes)` | 主分类器 |
  | **InceptionV3**<br>(Auxiliary) | 299×299 | `model.AuxLogits.fc` | `nn.Linear(768, num_classes)` | 辅助分类器，训练时同样需要替换 |

##### <font color="#00ffbf">**DenseNet简述**</font>
DenseNet（Densely Connected Convolutional Networks，密集连接卷积网络）是 2017 年 CVPR 的最佳论文，由康奈尔大学、清华大学和 Facebook 的研究团队提出。

如果把网络比作道路：
- **VGG** 是一条笔直的长路，一层接一层走到底
- **ResNet** 修了高速公路（skip connection），让梯度能抄近道
- **DenseNet** 则是**互通式立交桥**——每一层都和前面所有层直接相连

这意味着每一层都能"看到"并复用之前所有层学到的特征，信息不会白白丢弃，而是层层累积。好处是**参数量极少、精度很高**。但**理论很优美，工程有点贵**。


### **<font color="#39FF14" size=6 >ImageNet数据预处理</font>**

#### <font color="#CCFF00">**预训练模型的标准化参数**</font>

```python
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
```

<font color="#FF00A0">**重要**</font>：必须使用上述ImageNet参数，不要计算自己数据集的统计量！

In [22]:
# 完整的预处理配置
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize

transform = Compose([
    Resize(256),
    CenterCrop(224),
    ToTensor(),
    Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
print("预处理已配置！")

预处理已配置！


### **<font color="#39FF14" size=6 >特征数据集生成（加速训练）</font>**

#### <font color="#CCFF00">**为什么需要特征数据集？**</font>

在迁移学习中，前面的特征提取层是**冻结不动**的。这意味着无论你训练 10 个 epoch 还是 100 个 epoch，同一张图片经过这些层后，得到的特征向量**永远是一模一样的**。

想想看：如果每次训练都让整张图片从头再过一遍冻结层，就等于让一位已经定型的老师傅把同样的原料反复加工成相同的半成品——**白白浪费时间和算力**。

**更聪明的做法是**：

  1. **先存好半成品**：把整个训练集过一次冻结的特征提取层，把输出的特征向量保存下来（这就是"特征数据集"）。
  2. **只训练新员工**：用这个特征数据集去训练新的分类层，速度会快很多。
  3. **装回原模型**：分类层训练完成后，再把它接回原来的网络，得到最终可用的完整模型。

这样，原本需要在每个 epoch 重复做的大量前向传播，就被**一次性提前做完**了，后续训练分类层就像直接在"高级特征表格"上学习，省时又省力。

In [23]:
# 特征数据集生成函数
from torch.utils.data import TensorDataset

def preprocessed_dataset(model, loader, device=None):
    """
    通过冻结层生成特征数据集
    model: 特征提取器模型（冻结状态）
    loader: 数据加载器
    """
    if device is None:
        device = next(model.parameters()).device
    
    features = None
    labels = None
    
    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            output = model(x.to(device))
            if i == 0:
                features = output.cpu()
                labels = y.cpu()
            else:
                features = torch.cat([features, output.cpu()])
                labels = torch.cat([labels, y.cpu()])
    
    return TensorDataset(features, labels)

print("preprocessed_dataset函数定义完成！")

preprocessed_dataset函数定义完成！


#### <font color="#CCFF00">**Identity层的应用**</font>

为了提取倒数第二层的输出（特征），我们将最后一层替换为**Identity层**：
```python
alex.classifier[6] = nn.Identity()  # 替换为恒等映射
```

这样，模型的输出就是classifier[5]的特征（4096维），可以直接用于训练新的分类器。

#### <font color="#CCFF00">**微调策略详解**</font>

迁移学习要根据**数据集大小**和**与ImageNet的相似度**来选择策略：

| 场景 | 数据集大小 | 与ImageNet相似度 | 推荐策略 |
|:---|:---|:---|:---|
| **场景1** | 小（< 1万） | 高 | 冻结特征提取器，只训练分类器 |
| **场景2** | 小（< 1万） | 低 | 先冻结训练分类器；效果不佳时，再极小幅微调最后1-2层 |
| **场景3** | 大（> 10万） | 高 | 用较小学习率微调所有层 |
| **场景4** | 大（> 10万） | 低 | 从预训练权重开始微调所有层（完全从头训练一般没必要）|


**如何做到极小幅的微调：**"极小幅微调" ≈ 很小的学习率（1e-5） + 很短的训练时间（5-10 epoch） + 只动最后1-2层 + 必要时加大 weight decay + “SGD + Momentum“


<font color="#FF00A0" >**逐层微调（Layer-wise Fine-tuning）**</font>

越靠近输入的层学到的是越通用的特征（边缘、颜色），越靠近输出的层学到的是越任务相关的特征。所以可以**给不同层设置不同学习率**：



**PyTorch实现**（以 VGG / AlexNet 结构为例）：
```python
optim.Adam([
    {'params': model.features[:6].parameters(), 'lr': 0},       # 冻结
    {'params': model.features[6:12].parameters(), 'lr': 1e-5},  # 低学习率
    {'params': model.features[12:].parameters(), 'lr': 1e-4},  # 中学习率
    {'params': model.classifier.parameters(), 'lr': 1e-3},     # 高学习率
])
```

如果是 **ResNet** 等模型，层名不同，写法如下：
```python
optim.Adam([
    {'params': model.layer1.parameters(), 'lr': 1e-5},
    {'params': model.layer2.parameters(), 'lr': 1e-5},
    {'params': model.layer3.parameters(), 'lr': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3},
])
```

<font color="#FF00A0" >**学习率调度策略**</font>

迁移学习的训练通常分为两个阶段：

**阶段1：训练分类器（10-20 epochs）**
- 冻结特征提取器
- 学习率：`1e-3`
- 快速收敛

**阶段2：微调整个网络（可选，5-10 epochs）**
- 解冻所有层
- 学习率：`1e-5`（很小的学习率）
- 轻微调整特征

```python
# 两阶段训练示例

# 阶段1：只训练分类器
freeze_model(model)
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
train_model(model, train_loader, val_loader, optimizer, epochs=20)

# 阶段2：微调整个网络（可选）
unfreeze_model(model)
# 解冻后务必重新创建 optimizer，不要直接复用阶段1的实例！
optimizer = optim.Adam(model.parameters(), lr=1e-5)  # 小学习率
train_model(model, train_loader, val_loader, optimizer, epochs=10)
```

<font color="#FF00A0" >**常见的迁移学习误区**</font>

| 误区 | 正确做法 |
|:---|:---|
| ❌ 不冻结特征提取器，用大学习率训练整个网络 | ✅ 冻结特征提取器，只训练（新替换的）分类器 |
| ❌ 用同一个大学习率训练所有层 | ✅ 特征层用小学习率保护预训练知识，分类层可用较大学习率 |
| ❌ 忽略数据预处理（使用自己的mean/std） | ✅ 使用预训练模型的mean/std |
| ❌ 数据集小还微调所有层 | ✅ 数据集小应该冻结特征提取器 |
| ❌ 训练太多epoch导致过拟合 | ✅ 早停，监控验证集性能，务必注意看验证集“脸色“行事 |


### **<font color="#39FF14" size=6 >迁移学习总结</font>**

#### <font color="#CCFF00">**完整流程**</font>

1. **选择预训练模型**：如 ResNet18、AlexNet 等，在 ImageNet 上训练好的模型
2. **冻结特征提取层**：锁定预训练权重，使其不参与训练
3. **替换分类层**：将原来的 1000 类输出改为自己的类别数
4. **使用 ImageNet 标准化参数**：保证输入数据的分布与预训练时一致
5. **生成特征数据集（可选但推荐）**：让全部数据只过一次冻结层，提取特征向量并保存，后续训练分类器时大幅加速
6. **训练新的分类层**：只训练替换后的分类层，学习率可稍大（如 `1e-3`）
7. **（可选）微调**：若效果不满意，解冻最后 1~2 层或全部层，用很小学习率（如 `1e-5`）进一步微调
8. **评估与对比**：在测试集上验证效果，并与之前的模型进行对比


In [24]:
# ============================================================================
# 迁移学习完整示例：ResNet18 on RPS Dataset
# ============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import time

print("=" * 70)
print("Transfer Learning: ResNet18 on RPS Dataset")
print("=" * 70)

# ============================================================================
# Step 1: 数据准备（使用ImageNet统计量）
# ============================================================================

print("\n[Step 1] Data Preparation with ImageNet Normalization")

# ImageNet统计量（预训练模型的训练统计量）
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# 数据增强（与第一小节保持一致）
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)  # ImageNet统计量
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# 加载RPS数据集
train_dataset_raw = ImageFolder(root="./rps", transform=train_transform)
val_dataset_raw = ImageFolder(root="./rps", transform=val_transform)
test_dataset = ImageFolder(root="./rps-test-set", transform=val_transform)

# 划分训练/验证集（80/20，与第一小节一致）
from torch.utils.data import random_split, Subset
generator = torch.Generator().manual_seed(42)
train_size = int(0.8 * len(train_dataset_raw))
val_size = len(train_dataset_raw) - train_size
train_idx, val_idx = random_split(range(len(train_dataset_raw)), [train_size, val_size], generator=generator)

train_dataset = Subset(train_dataset_raw, train_idx.indices)
val_dataset = Subset(val_dataset_raw, val_idx.indices)

print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Test samples: {len(test_dataset)}")
print(f"  Classes: {train_dataset_raw.classes}")

# ============================================================================
# Step 2: 加载预训练ResNet18并冻结
# ============================================================================

print("\n[Step 2] Load Pre-trained ResNet18 and Freeze")

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"  Device: {device}")

# 加载预训练ResNet18
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 冻结所有层
for param in resnet18.parameters():
    param.requires_grad = False

# 替换分类层（原1000类 → 3类）
num_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_features, 3)

resnet18 = resnet18.to(device)

# 统计参数量
total_params = sum(p.numel() for p in resnet18.parameters())
trainable_params = sum(p.numel() for p in resnet18.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"  Total parameters: {total_params:,}")
print(f"  Frozen parameters: {frozen_params:,} ({frozen_params/total_params*100:.1f}%)")
print(f"  Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

# ============================================================================
# Step 3: 生成特征数据集（加速训练）
# ============================================================================

print("\n[Step 3] Generate Feature Dataset")

# 移除最后的FC层，获取特征提取器
feature_extractor = nn.Sequential(*list(resnet18.children())[:-1])
feature_extractor.eval()

def extract_features(dataset, batch_size=64):
    """提取冻结层的输出特征"""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    features_list = []
    labels_list = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            features = feature_extractor(images)
            features = features.view(features.size(0), -1)  # Flatten
            features_list.append(features.cpu())
            labels_list.append(labels)
    
    return torch.cat(features_list, dim=0), torch.cat(labels_list, dim=0)

start_time = time.time()
train_features, train_labels = extract_features(train_dataset)
val_features, val_labels = extract_features(val_dataset)
test_features, test_labels = extract_features(test_dataset)
extract_time = time.time() - start_time

print(f"  Feature extraction time: {extract_time:.2f}s")
print(f"  Feature shape: {train_features.shape}")
print(f"  Feature dimension: {train_features.shape[1]}")

# 创建特征数据集（只包含提取的特征和标签）
train_feature_dataset = TensorDataset(train_features, train_labels)
val_feature_dataset = TensorDataset(val_features, val_labels)
test_feature_dataset = TensorDataset(test_features, test_labels)

train_feature_loader = DataLoader(train_feature_dataset, batch_size=64, shuffle=True)
val_feature_loader = DataLoader(val_feature_dataset, batch_size=64, shuffle=False)
test_feature_loader = DataLoader(test_feature_dataset, batch_size=64, shuffle=False)

# ============================================================================
# Step 4: 训练新的分类层
# ============================================================================

print("\n[Step 4] Train New Classifier Layer")

# 只训练FC层
optimizer = optim.Adam(resnet18.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 使用特征数据集训练（速度更快）
def train_classifier(model, feature_loader, criterion, optimizer, device, epochs=20):
    model.train()
    history = {'loss': [], 'acc': []}
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        for features, labels in feature_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model.fc(features)  # 直接输入特征到FC层
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * features.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        epoch_loss = total_loss / total
        epoch_acc = correct / total
        history['loss'].append(epoch_loss)
        history['acc'].append(epoch_acc)
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch [{epoch+1:2d}/{epochs}] Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")
    
    return history

train_start = time.time()
history = train_classifier(resnet18, train_feature_loader, criterion, optimizer, device, epochs=20)
train_time = time.time() - train_start

print(f"  Training time: {train_time:.2f}s")

# ============================================================================
# Step 5: 模型评估（与第一小节对比）
# ============================================================================

print("\n" + "=" * 70)
print("Model Evaluation on Test Set")
print("=" * 70)

resnet18.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_feature_loader:
        features = features.to(device)
        outputs = resnet18.fc(features)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 计算指标
test_acc = accuracy_score(all_labels, all_preds)
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=train_dataset_raw.classes, digits=4))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
print(cm)

# 各类别统计
print("\nPer-class Statistics:")
for i, cls in enumerate(train_dataset_raw.classes):
    mask = all_labels == i
    class_acc = accuracy_score(all_labels[mask], all_preds[mask])
    print(f"  {cls}: {class_acc:.4f} ({class_acc*100:.2f}%)")

# ============================================================================
# Step 6: 与第一小节结果对比
# ============================================================================

print("\n" + "=" * 70)
print("Comparison: Custom CNN vs Transfer Learning (ResNet18)")
print("=" * 70)

# 第一小节的结果（用户提供）
custom_cnn_acc = 0.9462
custom_cnn_params = 406019  # 约40万

print(f"\n{'Metric':<30} {'Custom CNN':<20} {'ResNet18 TL':<20}")
print("-" * 70)
print(f"{'Total Parameters':<30} {custom_cnn_params:>19,} {total_params:>19,}")
print(f"{'Trainable Parameters':<30} {custom_cnn_params:>19,} {trainable_params:>19,}")
print(f"{'Test Accuracy':<30} {custom_cnn_acc:>19.4f} {test_acc:>19.4f}")

# 计算性能差异
acc_diff = (test_acc - custom_cnn_acc) * 100
param_ratio = trainable_params / custom_cnn_params

print(f"\nKey Findings:")
print(f"  • Transfer learning uses {param_ratio:.2%} trainable parameters of custom CNN")
print(f"  • Accuracy difference: {acc_diff:+.2f}%")
print(f"  • Feature extraction time: {extract_time:.2f}s")
print(f"  • Training time: {train_time:.2f}s")

if test_acc >= custom_cnn_acc:
    print(f"  ✓ Transfer learning achieves comparable or better accuracy with much fewer parameters!")
else:
    print(f"  Note: Custom CNN performs slightly better, but uses {1/param_ratio:.1f}x more trainable parameters")

print("\n" + "=" * 70)
print("Transfer Learning Complete!")
print("=" * 70)


Transfer Learning: ResNet18 on RPS Dataset

[Step 1] Data Preparation with ImageNet Normalization
  Training samples: 2016
  Validation samples: 504
  Test samples: 372
  Classes: ['paper', 'rock', 'scissors']

[Step 2] Load Pre-trained ResNet18 and Freeze
  Device: mps
  Total parameters: 11,178,051
  Frozen parameters: 11,176,512 (100.0%)
  Trainable parameters: 1,539 (0.0%)

[Step 3] Generate Feature Dataset
  Feature extraction time: 36.21s
  Feature shape: torch.Size([2016, 512])
  Feature dimension: 512

[Step 4] Train New Classifier Layer
  Epoch [ 5/20] Loss: 0.1046, Acc: 0.9821
  Epoch [10/20] Loss: 0.0577, Acc: 0.9911
  Epoch [15/20] Loss: 0.0392, Acc: 0.9945
  Epoch [20/20] Loss: 0.0288, Acc: 0.9965
  Training time: 1.48s

Model Evaluation on Test Set

Test Accuracy: 0.8763 (87.63%)

Classification Report:
              precision    recall  f1-score   support

       paper     1.0000    0.8145    0.8978       124
        rock     0.9806    0.8145    0.8899       124
    scis

In [ ]:
# ============================================================================
# 支持 Fine-tuning的迁移学习代码
# ============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, Subset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import time

print("=" * 70)
print("Transfer Learning: ResNet18 with Fine-tuning")
print("=" * 70)

# ============================================================================
# Step 1: 数据准备
# ============================================================================

print("\n[Step 1] Data Preparation")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# 加载数据集
train_dataset_raw = ImageFolder(root="./rps", transform=train_transform)
val_dataset_raw = ImageFolder(root="./rps", transform=val_transform)
test_dataset = ImageFolder(root="./rps-test-set", transform=val_transform)

# 划分训练/验证集
generator = torch.Generator().manual_seed(42)
train_size = int(0.8 * len(train_dataset_raw))
val_size = len(train_dataset_raw) - train_size
train_idx, val_idx = random_split(range(len(train_dataset_raw)), [train_size, val_size], generator=generator)

train_dataset = Subset(train_dataset_raw, list(train_idx))
val_dataset = Subset(val_dataset_raw, list(val_idx))

# DataLoader（注意：这里用原始图像，不是提取的特征）
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Test samples: {len(test_dataset)}")

# ============================================================================
# Step 2: 加载预训练模型并配置
# ============================================================================

print("\n[Step 2] Model Setup with Fine-tuning")

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"  Device: {device}")

# 加载预训练 ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 替换 FC 层为更深的分类头（修复容量不足问题）
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256, 3)
)

# ===== 关键：Fine-tuning 配置 =====
# 方案 A：只训练 FC 层（原来的方法，效果差）
# for param in model.parameters():
#     param.requires_grad = False
# for param in model.fc.parameters():
#     param.requires_grad = True

# 方案 B：Fine-tuning 最后两层 + FC（推荐）
# 冻结前面所有层
for param in model.parameters():
    param.requires_grad = False

# 解冻 layer4 和 layer3（最后两个残差块）
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.layer3.parameters():
    param.requires_grad = True

# 解冻 FC 层（包括新增的层）
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"  Total parameters: {total_params:,}")
print(f"  Frozen parameters: {frozen_params:,} ({frozen_params/total_params*100:.1f}%)")
print(f"  Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

# ============================================================================
# Step 3: 训练配置（使用不同的学习率）
# ============================================================================

print("\n[Step 3] Training Configuration")

criterion = nn.CrossEntropyLoss()

# 对不同层使用不同学习率（关键！）
# 预训练层用较小学习率，新层用较大学习率
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 1e-4},  # 预训练层：小学习率
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}       # 新层：大学习率
])

# 学习率衰减
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# ============================================================================
# Step 4: 训练循环（使用完整模型，不是提取的特征）
# ============================================================================

print("\n[Step 4] Training")

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)  # 完整前向传播
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return total_loss / total, correct / total

# 训练循环
epochs = 20
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

start_time = time.time()

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'resnet18_finetuned_best.pth')

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch [{epoch+1:2d}/{epochs}] "
            f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

train_time = time.time() - start_time
print(f"  Training time: {train_time:.2f}s")

# ============================================================================
# Step 5: 测试评估
# ============================================================================

print("\n" + "=" * 70)
print("Model Evaluation on Test Set")
print("=" * 70)

# 加载最佳模型
model.load_state_dict(torch.load('resnet18_finetuned_best.pth'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = accuracy_score(all_labels, all_preds)
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=train_dataset_raw.classes, digits=4))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
print(cm)

print("\nPer-class Statistics:")
for i, cls in enumerate(train_dataset_raw.classes):
    mask = all_labels == i
    class_acc = accuracy_score(all_labels[mask], all_preds[mask])
    print(f"  {cls}: {class_acc:.4f} ({class_acc*100:.2f}%)")

# ============================================================================
# Step 6: 对比
# ============================================================================

print("\n" + "=" * 70)
print("Comparison: Custom CNN vs Transfer Learning")
print("=" * 70)

custom_cnn_acc = 0.9462
custom_cnn_params = 406019
old_tl_acc = 0.8763
old_tl_params = 1539

print(f"\n{'Model':<25} {'Test Acc':<12} {'Trainable Params':<20}")
print("-" * 60)
print(f"{'Custom CNN':<25} {custom_cnn_acc:<12.4f} {custom_cnn_params:<20,}")
print(f"{'ResNet18 (Frozen)':<25} {old_tl_acc:<12.4f} {old_tl_params:<20,}")
print(f"{'ResNet18 (Fine-tuned)':<25} {test_acc:<12.4f} {trainable_params:<20,}")

if test_acc > custom_cnn_acc:
    print(f"\n✓ Fine-tuned ResNet18 outperforms Custom CNN by {(test_acc-custom_cnn_acc)*100:.2f}%!")
elif test_acc > old_tl_acc:
    print(f"\n✓ Fine-tuning improves accuracy by {(test_acc-old_tl_acc)*100:.2f}% over frozen approach")
else:
    print(f"\n⚠ Consider training longer or unfreezing more layers")

Transfer Learning: ResNet18 with Fine-tuning

[Step 1] Data Preparation
  Training samples: 2016
  Validation samples: 504
  Test samples: 372

[Step 2] Model Setup with Fine-tuning
  Device: mps
  Total parameters: 11,308,611
  Frozen parameters: 683,072 (6.0%)
  Trainable parameters: 10,625,539 (94.0%)

[Step 3] Training Configuration

[Step 4] Training
  Epoch [ 5/20] Train Loss: 0.0058, Acc: 0.9980 | Val Loss: 0.0003, Acc: 1.0000
  Epoch [10/20] Train Loss: 0.0153, Acc: 0.9950 | Val Loss: 0.0001, Acc: 1.0000
  Epoch [15/20] Train Loss: 0.0002, Acc: 1.0000 | Val Loss: 0.0000, Acc: 1.0000
  Epoch [20/20] Train Loss: 0.0001, Acc: 1.0000 | Val Loss: 0.0000, Acc: 1.0000
  Training time: 483.33s

Model Evaluation on Test Set

Test Accuracy: 0.9435 (94.35%)

Classification Report:
              precision    recall  f1-score   support

       paper     0.8552    1.0000    0.9219       124
        rock     1.0000    1.0000    1.0000       124
    scissors     1.0000    0.8306    0.9075     

 ### <font color="#FF6B00" size=6>**🤔分析：上面的输出结果说明什么？**</font>
 ￼

## <font color="#FFEA00" > **本章小结** </font>

#### **第1节：CNN神经网络训练完整流程**
  > - 使用ImageFolder加载按文件夹组织的图像数据，通过数据增强（翻转、旋转等）提升训练集多样性，验证/测试集仅做标准化。
  > - 网络采用"卷积块提取特征 → Global Average Pooling降维 → 极简全连接层"的结构，在保证表达能力的同时大幅减少参数量。
  > - 训练时使用Adam优化器配合学习率调度和早停机制，有效防止过拟合并自动保存最佳模型。

#### **第2节：现代经典CNN网络**
  > - VGG证明了小卷积核(3×3)堆叠的有效性，但参数量巨大；
  > - GoogLeNet引入Inception模块实现多尺度并行提取，配合1×1卷积降维和GAP，以1/20的参数量达到相近性能；
  > - ResNet通过残差连接解决深层网络退化问题，使网络可扩展至152+层，成为后续CNN架构的基础。
  
#### **第3节：迁移学习**

  > - 利用ImageNet预训练模型的通用特征，冻结特征提取层，仅替换并训练新的分类层即可快速适配新任务。
  > - 必须使用ImageNet的标准化参数保持数据分布一致。当冻结效果不佳时，可采用Fine-tuning策略：分层解冻网络，预训练层使用小学习率保护已有知识，新分类层使用较大学习率快速收敛。
  > - 这种方法在小数据集场景下能以极少训练成本获得接近甚至超越自定义CNN的性能。